# layout

> Stack, cluster, grid, and document flow

In [ ]:
#| default_exp layout

A component sets its own internal spacing. Its parent's layout sets how it sits beside other components. Here we define four relationships between children: a vertical stack, a wrapping row, an adaptive grid, and document flow. Each is an attribute that can go on an existing element. `Stack(...)` and `Section(..., stack=True)` use the same CSS, because the matching constructor puts the same attribute on a `Div`.

In [ ]:
from fasthtml.common import Div, H2, H3, P, Code, Small
from answerui.registry import hypelt, reg
from answerui.demo import example
import answerui.base  # the reset the spacing assumes, and the tokens it reads

In [ ]:
from fasthtml.common import Article, Header, Section, Ul, Li, to_xml
from answerui.demo import demo

## Stack

A heading, a description, and a note form a vertical group. Giving each child a bottom margin would leave an unwanted gap after the last one. Instead, the parent is a column with a `gap`. Its children stretch across the available width, with the gap between them. The default gap is `--space`.

The constructors accept CSS values for their sizing options, including token references. We put these in inline custom properties and leave the layout rules in the stylesheet. The helper `_layout` also keeps any ordinary `style` that the caller passes:

In [ ]:
def _layout(*c, props, style='', **attrs):
    style = ''.join(f'{k}: {v};' for k,v in props.items() if v is not None) + (style or '')
    return Div(*c, style=style or None, **attrs)

In [ ]:
@hypelt(sel='[stack]', layer='layout', css=r"""
--gap: var(--space); display: flex; flex-direction: column; gap: var(--gap);
""")
def Stack(
    *c,       # Children to arrange vertically
    gap=None, # CSS length or token reference; None uses --space
    **attrs,  # Attributes for the containing div
):
    "A vertical group with space between its children"
    return _layout(*c, props={'--gap': gap}, stack=True, **attrs)

None of these children has a margin. The stack supplies the spacing:

In [ ]:
@example(Stack)
def _stack_example():
    return Stack(H2('Account'), P('Manage your profile and preferences.'), Small('Changes apply to this workspace.'))

In [ ]:
Stack.demo()

```html
<div stack>
  <h2>Account</h2>
  <p>Manage your profile and preferences.</p>
<small>Changes apply to this workspace.</small></div>

```

For a tighter gap, pass `gap`. The constructor sets it as `--gap` on the same `div`, with no extra wrapper or styling class:

In [ ]:
to_xml(Stack(P('Ada Lovelace'), Small('Workspace owner'), gap='var(--space-2)'))

'<div stack style="--gap: var(--space-2);">\n  <p>Ada Lovelace</p>\n<small>Workspace owner</small></div>\n'

The `stack` attribute also works on a semantic element. Here the element stays a `section`, and the stylesheet gives it the stack's default gap:

In [ ]:
demo(Section(H2('Profile'), P('Visible to the rest of your team.'), stack=True))

```html
<section stack>
  <h2>Profile</h2>
  <p>Visible to the rest of your team.</p>
</section>

```

## Cluster

Tags and short pieces of metadata belong beside one another, but need to wrap when they run out of room. A cluster is a wrapping flex row, with its children centred on the cross axis. It uses the same gap in both directions and leaves each child at its natural width.

In [ ]:
@hypelt(sel='[cluster]', layer='layout', css=r"""
--gap: var(--space); display: flex; flex-wrap: wrap; align-items: center; gap: var(--gap);
""")
def Cluster(
    *c,       # Children to arrange in a wrapping row
    gap=None, # CSS length or token reference; None uses --space
    **attrs,  # Attributes for the containing div
):
    "A row of naturally sized children that wraps to fit"
    return _layout(*c, props={'--gap': gap}, cluster=True, **attrs)

In [ ]:
@example(Cluster)
def _cluster_example():
    return Cluster(Code('FastHTML'), Code('htmx'), Code('semantic HTML'), Code('CSS'), gap='var(--space-2)')

In [ ]:
Cluster.demo()

```html
<div cluster style="--gap: var(--space-2);">
<code>FastHTML</code><code>htmx</code><code>semantic HTML</code><code>CSS</code></div>

```

In a narrow container, the same cluster wraps without a breakpoint, even when the viewport is wide:

In [ ]:
demo(Div(_cluster_example(), style='max-inline-size: 14rem; border: 1px dashed var(--n-1)'))

```html
<div style="max-inline-size: 14rem; border: 1px dashed var(--n-1)">
  <div cluster style="--gap: var(--space-2);">
<code>FastHTML</code><code>htmx</code><code>semantic HTML</code><code>CSS</code>  </div>
</div>

```

When one child of a row should take the width left over, such as a name beside its actions, mark it `grow`. What follows it then sits at the far edge. Because a `grow` child can also shrink below its content's width, a long name wraps instead of pushing the actions out of the row:

In [ ]:
Cluster.rule('grow', '& > [grow] { flex: 1; min-inline-size: 0; }', layer='layout', doc='A child marked `grow` takes the width the others leave, so what follows it sits at the end of the row.')


Cluster.grow [layout] [cluster]: A child marked `grow` takes the width the others leave, so what follows it sits at the end of the row.

In [ ]:
demo(Cluster(Code('FastHTML'), Stack(P('Ada Lovelace'), Small('Owner'), gap='0', grow=True), Code('Edit'), Code('Remove')))


```html
<div cluster>
<code>FastHTML</code>  <div stack grow style="--gap: 0;">
    <p>Ada Lovelace</p>
<small>Owner</small>  </div>
<code>Edit</code><code>Remove</code></div>

```

## Grid

A cluster keeps each child at its natural width. A grid gives its children equal-width columns. We ask for as many columns as fit at a minimum width of `16rem`, then share the remaining space between them. Below that width, a single column shrinks to fit its container rather than overflowing. `min_width` changes the minimum column width without introducing breakpoints.

In [ ]:
@hypelt(sel='[grid]', layer='layout', css=r"""
--gap: var(--space); --grid-min: 16rem;
display: grid; gap: var(--gap);
grid-template-columns: repeat(auto-fit, minmax(min(100%, var(--grid-min)), 1fr));
> * { min-inline-size: 0; }
""")
def Grid(
    *c,             # Children to arrange in equal-width columns
    gap=None,       # CSS length or token reference; None uses --space
    min_width=None, # Minimum column width; None uses 16rem, capped at the container width
    **attrs,        # Attributes for the containing div
):
    "Equal-width columns whose count adapts to the available width"
    return _layout(*c, props={'--gap': gap, '--grid-min': min_width}, grid=True, **attrs)

Each column contains a stack with a smaller gap. Changing the grid's `gap` doesn't change the stacks inside it, because each layout sets its own `--gap`.

In [ ]:
@example(Grid)
def _grid_example():
    return Grid(
        Stack(H3('Inbox'), P('Requests waiting for a first look.'), gap='var(--space-2)'),
        Stack(H3('In progress'), P('Work the team has picked up.'), gap='var(--space-2)'),
        Stack(H3('Done'), P('Completed work, ready to share.'), gap='var(--space-2)'))

In [ ]:
Grid.demo()

```html
<div grid>
  <div stack style="--gap: var(--space-2);">
    <h3>Inbox</h3>
    <p>Requests waiting for a first look.</p>
  </div>
  <div stack style="--gap: var(--space-2);">
    <h3>In progress</h3>
    <p>Work the team has picked up.</p>
  </div>
  <div stack style="--gap: var(--space-2);">
    <h3>Done</h3>
    <p>Completed work, ready to share.</p>
  </div>
</div>

```

`grow` works in a stack too, where the child takes the height left over. That's useful for cards in a grid: mark the body `grow` and every card's last line sits at the bottom, however long its title is:

In [ ]:
Stack.rule('grow', '& > [grow] { flex: 1; }', layer='layout', doc='A child marked `grow` takes the height the others leave, so what follows it sits at the bottom of the stack.')

Stack.grow [layout] [stack]: A child marked `grow` takes the height the others leave, so what follows it sits at the bottom of the stack.

In [ ]:
demo(Grid(Stack(P('A short title'), P('Body', grow=True), Small('Foot')), Stack(P('A title long enough that it wraps onto a second line in a narrow column'), P('Body', grow=True), Small('Foot'))), width='24rem')

```html
<div grid>
  <div stack>
    <p>A short title</p>
    <p grow>Body</p>
<small>Foot</small>  </div>
  <div stack>
    <p>A title long enough that it wraps onto a second line in a narrow column</p>
    <p grow>Body</p>
<small>Foot</small>  </div>
</div>

```

## Lists

A list of things in the interface, such as the people who can open something or the rows of a table of links, is still a `ul`. As a layout rather than prose, it needs none of the bullets and indent that the base layer gives a prose list. A `ul` or `ol` with a `stack`, `cluster` or `grid` attribute has no markers or indent, and is laid out like any other container.

In [ ]:
reg.rule(':is(ul, ol):is([stack], [cluster], [grid]) { list-style: none; padding: 0; margin: 0; }', name='list', layer='layout',
    doc='A list that carries a layout is a layout, not prose: no markers, no indent.')

list [layout]: A list that carries a layout is a layout, not prose: no markers, no indent.

A stack of names, without the bullets a prose list would have:

In [ ]:
demo(Ul(Li('Ada Lovelace'), Li('Grace Hopper'), Li('Alan Turing'), stack=True, gap='var(--space-2)'))

```html
<ul stack gap="var(--space-2)">
  <li>Ada Lovelace</li>
  <li>Grace Hopper</li>
  <li>Alan Turing</li>
</ul>

```

## Flow

A sequence of headings and paragraphs needs a different rhythm. The space after a heading should be smaller than the space before the next section. A uniform flex or grid gap cannot express that distinction. Flow leaves its children in normal document flow. Its rule sets a top margin on each child after the first.

The default space is `1em`, measured in the font size of the child that gets the margin. A heading therefore gets more space above it than a paragraph does. The child right after a heading gets half the usual top margin. `Flow`'s `gap` sets `--flow-space`. Because the other layouts use `--gap` instead, a stack inside a flow can have its own internal gap without changing the margin above it. A flow directly inside another flow uses its own `--flow-space` for the margin above it, not its parent's, because the margin is computed on the child. Wrap the inner flow in a plain `Div`: the `Div` then gets the outer flow's spacing above it, and the inner flow's spacing applies only between its own children. For long-form articles, use the `prose` attribute: the base layer's [typrose](https://github.com/AnswerDotAI/typrose) rule styles their lists, quotes, code, and tables more fully.

In [ ]:
@hypelt(sel='[flow]', layer='layout', css=r"""
--flow-space: 1em;
> * + * { margin-block-start: var(--flow-space); }
> :is(h1, h2, h3, h4, h5, h6) + * { margin-block-start: calc(var(--flow-space) / 2); }
""")
def Flow(
    *c,       # Children in normal document flow
    gap=None, # CSS length or token reference; None uses 1em with half the space after headings
    **attrs,  # Attributes for the containing div
):
    "Document flow with closer spacing after headings"
    return _layout(*c, props={'--flow-space': gap}, flow=True, **attrs)

In [ ]:
@example(Flow)
def _flow_example():
    return Flow(H2('Sharing a workspace'), P('Invite your colleagues to work on the same projects.'),
        P('Everyone sees the latest changes.'), H3('Permissions'),
        P('Choose who can edit and who can view.'))

In [ ]:
Flow.demo()

```html
<div flow>
  <h2>Sharing a workspace</h2>
  <p>Invite your colleagues to work on the same projects.</p>
  <p>Everyone sees the latest changes.</p>
  <h3>Permissions</h3>
  <p>Choose who can edit and who can view.</p>
</div>

```

## Composition

The page below uses a stack to space its header, project grid, and help text. The header has its own smaller gap and a cluster of metadata. The article uses flow for its heading and paragraphs. Each layout spaces only its own direct children, whatever layouts its neighbours use.

In [ ]:
demo(Section(
    Header(H2('Workspace'), P('A shared place for the team\'s projects.'),
        Cluster(Small('3 projects'), Small('Updated today'), gap='var(--space-2)'),
        stack=True, style='--gap: var(--space-2)'),
    _grid_example(),
    Article(H3('Working together'), P('Pick a project to see its latest work.'),
        P('Use the inbox for requests that have not yet been assigned.'), flow=True),
    stack=True, style='--gap: var(--space2)'))

```html
<section stack style="--gap: var(--space2)">
  <header stack style="--gap: var(--space-2)">
    <h2>Workspace</h2>
    <p>A shared place for the team's projects.</p>
    <div cluster style="--gap: var(--space-2);">
<small>3 projects</small><small>Updated today</small>    </div>
  </header>
  <div grid>
    <div stack style="--gap: var(--space-2);">
      <h3>Inbox</h3>
      <p>Requests waiting for a first look.</p>
    </div>
    <div stack style="--gap: var(--space-2);">
      <h3>In progress</h3>
      <p>Work the team has picked up.</p>
    </div>
    <div stack style="--gap: var(--space-2);">
      <h3>Done</h3>
      <p>Completed work, ready to share.</p>
    </div>
  </div>
  <article flow>
    <h3>Working together</h3>
    <p>Pick a project to see its latest work.</p>
    <p>Use the inbox for requests that have not yet been assigned.</p>
  </article>
</section>

```

Here the plain `Div` gets a top margin of `2rem` from the outer flow. Without the `Div`, the inner flow's `.25rem` would set that top margin:

In [ ]:
demo(Flow(P('Before the nested flow.'), Div(Flow(P('Inner one.'), P('Inner two.'), gap='.25rem')), P('After it.'), gap='2rem'))

```html
<div flow style="--flow-space: 2rem;">
  <p>Before the nested flow.</p>
  <div>
    <div flow style="--flow-space: .25rem;">
      <p>Inner one.</p>
      <p>Inner two.</p>
    </div>
  </div>
  <p>After it.</p>
</div>

```

To change a layout's defaults across an application, register CSS in the `app` layer, for example `[cluster] { --gap: var(--space-2); }`. For one element, pass the constructor's option, such as `gap`, or set the custom property in `style`. Other adjustments, such as centring a particular cluster with `justify-content`, remain ordinary CSS. Layouts add no background, border, padding, or outer margin of their own.

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()